# PASSO 0 — Exploração Visual Interativa
## MTESC04_NOCI — Sessão 1 (08/07/2024)

**Objetivo:** navegar pelo LFP dos 3 .ns2 juntos, ver onde tem teta (4–8 Hz) e gama (30–80 Hz), marcar instantes e enviar ao pipeline. Este passo vem ANTES da triagem automática.

**Controles:** use o seletor de canal + slider de tempo, clique em Atualizar, marque com o botão verde, e use a Célula 4 para calcular PAC imediato.

⚠️ Se o pipeline (etapas 1–7) não achar acoplamento mas você vê teta+gama aqui: você tem evidência documentada. Marque o instante.

In [5]:
# CELULA 1: CONFIGURACAO E CARREGAMENTO
import os, sys
import numpy as np
from scipy import signal

SCRIPT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, SCRIPT_DIR)
from pipeline.ns2_utils import concatena_sessao

PASTA_BASAL = r"C:\acoplamento_theta-gamma\MTESC04_NOCI\MTESC04 -- 1 - infusao - 08-07-2024\Basal antes da infusao"
SESSAO = "MTESC04_S1"
CANAL_DEFAULT = 5
FS = 1000.0
JANELA_S = 10

print("Carregando sessao...")
dados, fs, ids_canais, offsets = concatena_sessao(PASTA_BASAL, fs_esperado=FS)
n_samples, n_canais = dados.shape
duracao_s = n_samples / fs
print(f"  {n_canais} canais x {n_samples:,} amostras ({duracao_s:.0f} s, {duracao_s/60:.1f} min)")
dados = dados.astype(np.float64)

def aplica_notch(sinal, fs, linha_hz=60.0, Q=30):
    b, a = signal.iirnotch(linha_hz, Q, fs)
    return signal.filtfilt(b, a, sinal)

def filtra_butter(sinal, low, high, fs, order=3):
    sos = signal.butter(order, [low, high], btype='band', fs=fs, output='sos')
    return signal.sosfiltfilt(sos, sinal)

BANDA_TETA = (4, 8)
BANDA_GAMMA = (30, 80)
DS_FACTOR = max(1, int(fs // 200))
DS_FS = fs / DS_FACTOR
print(f"  Downsample: {fs} -> {DS_FS} Hz (fator {DS_FACTOR})")
print("Dados carregados. Execute a Celula 2.")

global_state = dict(
    dados=dados, fs=fs, fs_ds=DS_FS, ds_factor=DS_FACTOR,
    n_samples=n_samples, n_canais=n_canais,
    ids_canais=ids_canais, duracao_s=duracao_s,
    canal_atual=CANAL_DEFAULT,
    t_inicio=0.0, t_fim=min(60, duracao_s),
    candidatos=[]
)

Carregando sessao...
  32 canais x 874,376 amostras (874 s, 14.6 min)
  Downsample: 1000.0 -> 200.0 Hz (fator 5)
Dados carregados. Execute a Celula 2.


In [6]:
# CELULA 2: VISUALIZADOR INTERATIVO (bqplot)
import warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from IPython.display import display, clear_output
import bqplot as bq
import numpy as np

gs = global_state
t_total = gs['duracao_s']

# --- scales ---
sc_x        = bq.LinearScale(min=0, max=t_total)
sc_raw      = bq.LinearScale(min=-3, max=3)
sc_theta    = bq.LinearScale(min=-2, max=2)
sc_gamma    = bq.LinearScale(min=-2, max=2)
sc_spec_y   = bq.LinearScale(min=0, max=100)
sc_psd      = bq.LinearScale(min=0, max=50)
sc_psd_x    = bq.LinearScale(min=0, max=100)
sc_color    = bq.ColorScale(scheme='viridis')

# --- LFP lines ---
line_raw = bq.Lines(x=[], y=[], scales={'x': sc_x, 'y': sc_raw},
                  colors=['#1a1a2e'], line_width=0.8, labels=['LFP bruto'])
line_theta = bq.Lines(x=[], y=[], scales={'x': sc_x, 'y': sc_theta},
                     colors=['#e63946'], line_width=1.2, labels=['theta 4-8 Hz'],
                     display_legend=True)
line_gamma = bq.Lines(x=[], y=[], scales={'x': sc_x, 'y': sc_gamma},
                     colors=['#457b9d'], line_width=1.0, labels=['gamma 30-80 Hz'],
                     display_legend=True)
ax_lfp_y = bq.Axis(scale=sc_raw, orientation='vertical', label='LFP (uV)', grid_lines='solid')
ax_lfp_x = bq.Axis(scale=sc_x,  orientation='horizontal', label='Tempo (s)', grid_lines='solid')
fig_lfp = bq.Figure(
    marks=[line_raw, line_theta, line_gamma],
    axes=[ax_lfp_x, ax_lfp_y],
    title='LFP - Raw + theta + gamma',
    layout=widgets.Layout(width='100%', height='220px'),
    interaction=bq.interacts.PanZoom(scales={'x': [sc_x]}),
    legend_location='top-right'
)

# --- spectrogram heatmap ---
# x,y must be 1D; color must be 2D (n_bins x n_freqs)
ax_spec_x = bq.Axis(scale=sc_x,  orientation='horizontal', label='Tempo (s)', grid_lines='solid')
ax_spec_y = bq.Axis(scale=sc_spec_y, orientation='vertical',  label='Freq (Hz)', grid_lines='solid')
heat_spec = bq.HeatMap(
    x=np.array([0.0, 1.0]),
    y=np.array([0.0, 1.0]),
    color=np.array([[-80.0, -60.0], [-80.0, -60.0]]),
    scales={'x': sc_x, 'y': sc_spec_y, 'color': sc_color}
)
fig_spec = bq.Figure(
    marks=[heat_spec],
    axes=[ax_spec_x, ax_spec_y],
    title='Espectrograma (60 Hz notch)',
    layout=widgets.Layout(width='100%', height='180px'),
    interaction=bq.interacts.PanZoom(scales={'x': [sc_x]})
)

# --- PSD ---
ax_psd_x = bq.Axis(scale=sc_psd_x, orientation='horizontal', label='Freq (Hz)', grid_lines='solid')
ax_psd_y = bq.Axis(scale=sc_psd,     orientation='vertical',  label='Potencia (uV^2/Hz)', grid_lines='solid')
band_tl = bq.Lines(x=[4,4],  y=[0,200], scales={'x':sc_psd_x,'y':sc_psd}, colors=['#e6394644'], line_width=3)
band_tr = bq.Lines(x=[8,8],  y=[0,200], scales={'x':sc_psd_x,'y':sc_psd}, colors=['#e6394644'], line_width=3)
band_gl = bq.Lines(x=[30,30],y=[0,200], scales={'x':sc_psd_x,'y':sc_psd}, colors=['#457b9d44'], line_width=3)
band_gr = bq.Lines(x=[80,80],y=[0,200], scales={'x':sc_psd_x,'y':sc_psd}, colors=['#457b9d44'], line_width=3)
line_psd = bq.Lines(x=[], y=[], scales={'x': sc_psd_x, 'y': sc_psd},
                  colors=['#2d6a4f'], line_width=2, labels=['PSD'], display_legend=True)
fig_psd = bq.Figure(
    marks=[band_tl, band_tr, band_gl, band_gr, line_psd],
    axes=[ax_psd_x, ax_psd_y],
    title='PSD (Welch, 0-100 Hz)',
    layout=widgets.Layout(width='100%', height='180px'),
    interaction=bq.interacts.PanZoom(scales={'x': [sc_psd_x]}),
    legend_location='top-right'
)

# --- widgets ---
ch_options = list(range(gs['n_canais']))
ch_dropdown = widgets.Dropdown(options=ch_options, value=gs['canal_atual'],
                              description='Canal:', style={'description_width':'50px'},
                              layout=widgets.Layout(width='90px'))
ch_label = widgets.HTML(value=f'<b>{gs["ids_canais"][gs["canal_atual"]]}</b>')
t_max = gs['duracao_s']
t_slider_ini = widgets.FloatSlider(value=0.0, min=0.0, max=max(0, t_max-JANELA_S),
                                   step=1.0, description='Inicio (s):', readout=True,
                                   style={'description_width':'80px'},
                                   layout=widgets.Layout(width='340px'))
t_slider_fim = widgets.FloatSlider(value=JANELA_S, min=JANELA_S, max=t_max,
                                   step=1.0, description='Fim (s):', readout=True,
                                   style={'description_width':'80px'},
                                   layout=widgets.Layout(width='340px'))
show_theta = widgets.Checkbox(value=True, description='theta 4-8 Hz', style={'description_width':'initial'})
show_gamma = widgets.Checkbox(value=True, description='gamma 30-80 Hz', style={'description_width':'initial'})
btn_update = widgets.Button(description='Atualizar', button_style='primary',
                           layout=widgets.Layout(width='110px'))
btn_marcar = widgets.Button(description='Marcar instante', button_style='success',
                           layout=widgets.Layout(width='150px'))
btn_limpar = widgets.Button(description='Limpar', button_style='warning',
                           layout=widgets.Layout(width='110px'))
btn_salvar = widgets.Button(description='Salvar CSV', button_style='info',
                           layout=widgets.Layout(width='120px'))
log_out = widgets.Output(layout=widgets.Layout(border='1px solid #ccc', padding='8px',
                                              width='100%', height='160px', overflow_y='scroll'))

def build_traces(canal, t0, t1):
    i0, i1 = int(t0*gs['fs']), int(t1*gs['fs'])
    i0, i1 = max(0,i0), min(gs['n_samples'],i1)
    idx_ds = slice(i0, i1, gs['ds_factor'])
    t_ds = np.arange(i1-i0)[idx_ds] / gs['fs_ds'] + t0
    raw_ds = (gs['dados'][idx_ds, canal]).astype(np.float32)
    theta_ds = filtra_butter(raw_ds, *BANDA_TETA, gs['fs_ds'], order=3)
    gamma_ds = filtra_butter(raw_ds, *BANDA_GAMMA, gs['fs_ds'], order=3)
    r_raw = float(np.nanmax(np.abs(raw_ds))) * 1.1
    r_th  = float(np.nanmax(np.abs(theta_ds))) * 1.1
    r_gm  = float(np.nanmax(np.abs(gamma_ds))) * 1.1
    rmax = max(r_raw, r_th, r_gm)
    sc_raw.min   = sc_theta.min   = sc_gamma.min   = -rmax
    sc_raw.max   = sc_theta.max   = sc_gamma.max   =  rmax
    return dict(t=t_ds, raw=raw_ds, theta=theta_ds, gamma=gamma_ds)

def build_spectrogram(canal, t0, t1, nperseg=256, noverlap=200):
    i0, i1 = int(t0*gs['fs']), int(t1*gs['fs'])
    i0, i1 = max(0,i0), min(gs['n_samples'],i1)
    seg = gs['dados'][i0:i1, canal].astype(np.float64)
    seg = aplica_notch(seg, gs['fs'], 60.0)
    freqs, times, Sxx = signal.spectrogram(seg, fs=gs['fs'],
                                          nperseg=nperseg, noverlap=noverlap, scaling='density')
    mask = freqs <= 100
    return times + t0, freqs[mask], 10*np.log10(Sxx[mask, :] + 1e-10)

def build_psd(canal, t0, t1):
    i0, i1 = int(t0*gs['fs']), int(t1*gs['fs'])
    i0, i1 = max(0,i0), min(gs['n_samples'],i1)
    seg = gs['dados'][i0:i1, canal].astype(np.float64)
    seg = aplica_notch(seg, gs['fs'], 60.0)
    freqs, psd = signal.welch(seg, fs=gs['fs'], nperseg=min(1024, i1-i0))
    mask = freqs <= 100
    return freqs[mask], psd[mask]

def update_figures(b=None):
    canal = ch_dropdown.value
    t0, t1 = float(t_slider_ini.value), float(t_slider_fim.value)
    t1 = max(t1, t0 + 0.5)
    t_slider_fim.value = t1
    gs['canal_atual'] = canal
    gs['t_inicio'] = t0
    gs['t_fim']    = t1
    ch_label.value = f'<b>{gs["ids_canais"][canal]}</b> (canal {canal})'
    tr = build_traces(canal, t0, t1)
    line_raw.x   = tr['t'];   line_raw.y   = tr['raw']
    line_theta.x = tr['t'];   line_theta.y = tr['theta'] if show_theta.value else []
    line_gamma.x = tr['t'];   line_gamma.y = tr['gamma'] if show_gamma.value else []
    t_spec, freqs_spec, Sxx_db = build_spectrogram(canal, t0, t1)
    # HeatMap x,y must be 1D; color must be 2D (n_bins x n_freqs)
    heat_spec.x = t_spec.tolist()
    heat_spec.y = freqs_spec.tolist()
    heat_spec.color = Sxx_db.tolist()
    sc_spec_y.min = 0
    sc_spec_y.max = float(freqs_spec.max())
    sc_color.min  = float(np.nanmin(Sxx_db))
    sc_color.max  = float(np.nanmax(Sxx_db))
    freqs_psd, psd_vals = build_psd(canal, t0, t1)
    line_psd.x = freqs_psd.tolist()
    line_psd.y = psd_vals.tolist()
    sc_psd.max  = float(np.nanmax(psd_vals)) * 1.2
    sc_x.min, sc_x.max = t0, t1
    with log_out:
        print(f"  Canal {canal} ({gs['ids_canais'][canal]}) | t = {t0:.1f}-{t1:.1f}s | n = {int((t1-t0)*gs['fs']):,}")

def marcar_instante(b):
    canal = gs['canal_atual']
    t0, t1 = gs['t_inicio'], gs['t_fim']
    dur = t1 - t0
    entrada = {'rotulo': f'ch{canal}_{t0:.0f}s',
               't_start': round(t0,1), 't_end': round(t1,1),
               'canal': canal, 'duracao_s': round(dur,1), 'observacao': ''}
    gs['candidatos'].append(entrada)
    with log_out:
        print(f"  Marcado: ch{canal} @ {t0:.1f}-{t1:.1f}s ({dur:.0f}s) | Total: {len(gs['candidatos'])}")

def limpar_marcacoes(b):
    gs['candidatos'] = []
    with log_out:
        print("  Marcacoes limpas.")

def salvar_csv(b):
    if not gs['candidatos']:
        with log_out:
            print("  Nenhum candidato marcado.")
        return
    import pandas as pd
    df = pd.DataFrame(gs['candidatos'])
    cols = ['rotulo','t_start','t_end','canal','duracao_s','z_pico','fase_pico_hz','amp_pico_hz','observacao']
    df = df[[c for c in cols if c in df.columns]]
    out_path = os.path.join(os.getcwd(), 'candidatos.csv')
    df.to_csv(out_path, index=False)
    with log_out:
        print(f"  Salvo: {out_path}")
        print(df.to_string(index=False))

btn_update.on_click(update_figures)
btn_marcar.on_click(marcar_instante)
btn_limpar.on_click(limpar_marcacoes)
btn_salvar.on_click(salvar_csv)
update_figures()

display(widgets.VBox([
    widgets.HBox([ch_dropdown, ch_label, btn_update]),
    widgets.HBox([show_theta, show_gamma]),
    widgets.VBox([t_slider_ini, t_slider_fim]),
    fig_lfp,
    fig_spec,
    fig_psd,
    widgets.HBox([btn_marcar, btn_limpar, btn_salvar]),
    widgets.HTML('<hr>'),
    widgets.HTML('<b>Log de eventos:</b>'),
    log_out
]))

print("Visualizador pronto. Use os controles para navegar e marque instantes.")

Visualizador pronto. Use os controles para navegar e marque instantes.


In [ ]:
# CELULA 3: PSD RAPIDA DE TODOS OS CANAIS (visao geral)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(8, 4, figsize=(16, 18), sharex=True)
fs_ = gs['fs']

for ch in range(gs['n_canais']):
    ax = axes.flat[ch]
    seg = gs['dados'][:, ch].astype(np.float64)
    seg = aplica_notch(seg, fs_, 60.0)
    freqs, psd = signal.welch(seg, fs=fs_, nperseg=min(2048, len(seg)))
    ax.psd(psd, Fs=freqs, NFFT=min(1024,len(psd)))
    ax.set_title(f'ch{ch} {gs["ids_canais"][ch]}', fontsize=8)
    ax.set_xlim(0, 100)
    for band, lo, hi, color in [('teta', 4, 8, '#e63946'), ('gama', 30, 80, '#457b9d')]:
        ax.axvspan(lo, hi, alpha=0.15, color=color, label=band)

plt.suptitle(f'PSD todos canais — {SESSAO}', y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# CELULA 4B: CALCULO DIRETO DE PAC NO NOTEBOOK (KL-MI + surrogates)
import numpy as np
from scipy import signal
from scipy.stats import entropy

FASES_FREQ = np.arange(4, 15, 1)   # 4-14 Hz (teta)
AMPS_FREQ  = np.arange(30, 105, 5)  # 30-100 Hz (gama)
N_BINS = 18
N_SURR = 200

def _mi_kl(p):
    """KL-based MI from Tort et al. 2010."""
    p = p / (p.sum() + 1e-12)
    m  = p.mean()
    mi = entropy(p + 1e-12) - entropy(m + 1e-12)
    return mi

def _mi_celula(sinal, fs, f_ph, f_amp, n_bins=18):
    detr = signal.detrend(sinal)
    bp_ph  = filtra_butter(detr, f_ph-1,  f_ph+1,  fs)
    bp_amp = filtra_butter(detr, f_amp-5, f_amp+5, fs)
    fase = np.angle(signal.hilbert(bp_ph))
    amp  = np.abs(signal.hilbert(bp_amp))
    bins = np.linspace(-np.pi, np.pi, n_bins+1)
    p = np.zeros(n_bins)
    for b in range(n_bins):
        mask = (fase >= bins[b]) & (fase < bins[b+1])
        if mask.sum() > 0:
            p[b] = amp[mask].mean()
    if p.sum() == 0:
        return np.nan
    return _mi_kl(p)

def mi_mapa(sinal, fs, fases_freq, amps_freq):
    mapa = np.empty((len(fases_freq), len(amps_freq)))
    for i, fp in enumerate(fases_freq):
        for j, fa in enumerate(amps_freq):
            mapa[i, j] = _mi_celula(sinal, fs, fp, fa)
    return mapa

def z_comod(sinal, fs, fases_freq, amps_freq, n_surr=200, seed=42):
    """Compute z-score of observed MI vs surrogates."""
    mi_obs = mi_mapa(sinal, fs, fases_freq, amps_freq)
    n = len(sinal)
    shift = max(1, int(fs))
    rng = np.random.default_rng(seed)
    mi_surr = np.empty((n_surr,) + mi_obs.shape)
    for k in range(n_surr):
        off = int(rng.integers(shift, n - shift))
        mi_surr[k] = mi_mapa(np.concatenate([sinal[off:], sinal[:off]]),
                               fs, fases_freq, amps_freq)
    mu = np.nanmean(mi_surr, axis=0)
    sd = np.nanstd(mi_surr, axis=0)
    sd[sd < 1e-12] = 1e-12
    z = (mi_obs - mu) / sd
    idx = np.nanargmax(z)
    i, j = np.unravel_index(idx, z.shape)
    return z, float(z[i,j]), int(fases_freq[i]), int(amps_freq[j]), mi_obs

# --- rodar em cada candidato marcado ---
if not gs['candidatos']:
    print("Nenhum candidato marcado. Use a Celula 2 para marcar instantes primeiro.")
else:
    results = []
    for c in gs['candidatos']:
        ch = c['canal']
        t0, t1 = c['t_start'], c['t_end']
        i0 = int(t0 * gs['fs'])
        i1 = int(t1 * gs['fs'])
        seg = gs['dados'][i0:i1, ch].astype(np.float64)
        seg = aplica_notch(seg, gs['fs'], 60.0)
        print(f"Processando {c['rotulo']} ...", end=' ', flush=True)
        z, z_pico, f_ph, f_amp, mi_obs = z_comod(seg, gs['fs'], FASES_FREQ, AMPS_FREQ,
                                                   n_surr=N_SURR, seed=42)
        c.update({'z_pico': round(z_pico,3), 'fase_pico_hz': f_ph,
                  'amp_pico_hz': f_amp, 'z_mapa': z})
        results.append(c)
        print(f"z_pico={z_pico:.2f} @ {f_ph}x{f_amp} Hz")
    print(f"\nTotal: {len(results)} candidato(s) processado(s).")
    print("Execute a Celula 5 para visualizar os mapas PAC.")

In [ ]:
# CELULA 5: VISUALIZAR MAPAS PAC DOS CANDIDATOS MARCADOS
import matplotlib.pyplot as plt

cands_com_z = [c for c in gs['candidatos'] if 'z_mapa' in c]
if not cands_com_z:
    print("Nenhum mapa PAC disponivel. Execute a Celula 4B primeiro.")
else:
    n = len(cands_com_z)
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows), squeeze=False)
    fases = np.arange(4, 15, 1)
    amps  = np.arange(30, 105, 5)
    for i, c in enumerate(cands_com_z):
        ax = axes.flat[i]
        zm = c['z_mapa']
        vmax = max(3.0, float(np.nanmax(np.abs(zm))))
        im = ax.pcolormesh(fases, amps, zm.T, cmap='viridis', vmin=-vmax, vmax=vmax, shading='auto')
        ax.scatter([c['fase_pico_hz']], [c['amp_pico_hz']], s=120, c='red',
                   marker='X', edgecolors='white', linewidths=1.5, zorder=5)
        ax.set_xlabel('Fase (Hz)')
        ax.set_ylabel('Amplitude (Hz)')
        ax.set_title(f"ch{c['canal']} @ {c['t_start']:.0f}-{c['t_end']:.0f}s | "
                     f"z={c['z_pico']:.2f} @ {c['fase_pico_hz']}x{c['amp_pico_hz']}Hz", fontsize=10)
        plt.colorbar(im, ax=ax, label='z')
    for k in range(n, rows*cols):
        axes.flat[k].axis('off')
    fig.suptitle(f'Mapas PAC dos candidatos - {SESSAO}', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
    print(f"{n} mapa(s) plotado(s).")